<a href="https://colab.research.google.com/github/piercegregory-spec/python-exercises/blob/main/benelux_hercher_cuda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Benelux pairs (Erdős #850) on a GPU, prime-power sieve version: `benelux_hercher_cuda.ipynb`

A faster rewrite of `benelux_cuda.ipynb` for **Google Colab** (Runtime → Change runtime type → a GPU such as A100, L4 or T4). The notebook is self-contained.

**Definitions.** rad(x) = product of the distinct primes dividing x. A *first-kind* pair is m < n with rad(m) = rad(n) and rad(m+1) = rad(n+1). A *second-kind* pair has rad(m) = rad(n+1) and rad(m+1) = rad(n). Known: two infinite families plus the exceptional pairs **(75, 1215)** and **(35, 4374)**.

### Why this is faster than `benelux_cuda.ipynb`

`benelux_cuda` computes the exact radical of *every* integer (sieving with every prime ≤ √N, about 3 global atomics + 3 scattered writes per integer, then 64-bit division loops). But almost every n can be rejected without knowing its radical:

**Lemma.** Let r = rad(n), s = rad(n+1). If n has a partner m < n (either kind), then r·s ≤ 2n.
*Proof.* First kind: m ≡ 0 (mod r), m ≡ −1 (mod s), and n satisfies the same congruences, so rs | n − m, hence rs ≤ n. Second kind: m ≡ 0 (mod s), m ≡ −1 (mod r), so rs | m + n + 1, and 0 < m + n + 1 ≤ 2n, hence rs ≤ 2n. ∎

Writing E(x) = x / rad(x) = ∏ p^(e−1), the condition r·s ≤ 2n is the same as **E(n)·E(n+1) ≥ (n+1)/2**. And log₂E(x) = Σ log₂p over all prime powers p^k ∣ x with **k ≥ 2**. That is Hercher's Algorithm 1 sieve: it only touches multiples of p², p³, …, which is Σ 1/(p(p−1)) ≈ **0.77 updates per integer** (vs ≈ 3 for a full prime sieve), and it needs no per-number divisions.

Only about 5×10⁻⁵ of integers survive this filter at 10⁷ (and the fraction keeps falling). The survivors get their exact radicals on the CPU, then the same CRT partner test and verification as before.

### Exactness (no false negatives)

* Each prime power p^k contributes the integer weight w = ⌊2²⁰·log₂p⌋ + 1, a strict **upper bound**, so the sieve sum U(x) ≥ 2²⁰·log₂E(x) exactly (integer atomics, no rounding drift).
* The threshold uses the *smallest* n of each 8192-wide tile minus a margin of 4096 units (≈ 0.004 in log₂), so it is a **lower bound** for every n in the tile.
* So every n with E(n)E(n+1) ≥ (n+1)/2 is kept. Extra survivors only cost a little CPU time.
* Survivors are checked exactly. Radicals come from trial division to ∛x (the leftover cofactor is 1, q, q·q′ or q²). Every reported pair is then re-confirmed by full trial division to √x, and the run raises an error on any mismatch.

### GPU layout per block of B = 2²⁴ integers

| Kernel | One GPU thread = | Job |
|---|---|---|
| `k_zero` | one integer | clear the accumulator |
| `k_big` | (prime power q > 8192, chunk of 64 multiples) | global `atomicAdd` of w(q); fewer than 0.01 updates per integer |
| `k_tile` | one of 256 threads per 8192-integer tile | load the tile into shared memory, add the small prime powers (q ≤ 8192) with shared-memory atomics, apply the threshold, emit survivors |

The GPU is only synchronised at checkpoints (every 16 blocks), and only the few survivors are copied back.

> **Status:** the kernels were validated with Numba's CUDA *simulator* (against brute force and an exact-radical CPU reference); real-GPU speed has **not** been measured yet. Run the self-test cell first, then time 10⁹ before committing to a long run.


## 1. Check the GPU

In [ ]:
!nvidia-smi
import numba
from numba import cuda
print("numba", numba.__version__, "| CUDA available:", cuda.is_available())
if cuda.is_available():
    print(cuda.get_current_device().name)

Fri Oct  9 21:00:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             56W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Implementation (kernels + driver)

Defines the prime-power tables, the three GPU kernels, the CPU routines for the rare survivors, and `search_hercher(lo, hi, B, outdir)`, which returns `(first_kind, second_kind)` as sorted lists of `(m, n)`.

In [ ]:
import math
import os
import time

import numpy as np
from numba import cuda, njit, int32, int64

# ------------------------------------------------------------------ settings
TILE = 8192          # integers per shared-memory tile (one CUDA block per tile)
TPB = 256            # threads per CUDA block in the tile kernel
TPB_BIG = 128        # threads per CUDA block in the large prime-power kernel
K_CHUNK = 64         # multiples of one large prime power handled by one thread
SCALE = 1 << 20      # fixed-point scale: weights are log2(p) * SCALE (rounded UP)
MARGIN = 1 << 12     # safety margin on the threshold (~0.004 in log2 units)


def primes_upto(limit):
    sieve = np.ones(limit + 1, dtype=np.bool_
)
    sieve[:2] = False
    for i in range(2, int(limit ** 0.5) + 1):
        if sieve[i]:
            sieve[i * i::i] = False
    return np.nonzero(sieve)[0].astype(np.int64)


def prime_power_table(top):
    """All q = p^k with k >= 2 and q <= top, with weight w(q) = floor(SCALE*log2 p) + 1.

    w(q) is a strict upper bound for SCALE*log2(p), so summing w over all p^k | x
    (k >= 2) gives an upper bound for SCALE*log2(E(x)), E(x) = x / rad(x)."""
    qs, ws = [], []
    for p in primes_upto(math.isqrt(top) + 1).tolist():
        w = math.floor(SCALE * math.log2(p)) + 1
        q = p * p
        while q <= top:
            qs.append(q)
            ws.append(w)
            q *= p
    qs = np.array(qs, dtype=np.int64)
    ws = np.array(ws, dtype=np.int32)
    order = np.argsort(qs, kind="stable")
    return qs[order], ws[order]


# ---------------------------------------------------------------- GPU kernels
@cuda.jit
def k_zero(acc, size):
    i = cuda.grid(1)
    if i < size:
        acc[i] = 0


@cuda.jit
def k_big(L, size, item_q, item_c, item_w, nitems, acc):
    """Prime powers q > TILE. One thread = (q, chunk of K_CHUNK multiples).
    Adds w(q) to acc[j] for every L+j in [L, L+size) divisible by q."""
    t = cuda.grid(1)
    if t >= nitems:
        return
    q = item_q[t]
    w = item_w[t]
    j = ((L + q - 1) // q) * q - L + np.int64(item_c[t]) * K_CHUNK * q
    for _ in range(K_CHUNK):
        if j >= size:
            break
        cuda.atomic.add(acc, j, w)
        j += q


@cuda.jit
def k_tile(L, nn, size, acc, sq, sw, nsmall, out, ocount, cap, dbg, use_dbg):
    """One CUDA block = one tile of TILE consecutive n (plus n+1 of the last one).
    Loads the large-prime-power sums, adds the small prime powers (q <= TILE) in
    shared memory, then keeps n only if  U(n) + U(n+1) >= SCALE*log2((n+1)/2)."""
    sh = cuda.shared.array(8193, dtype=int32)
    soff = cuda.shared.array(TPB, dtype=int64)
    tid = cuda.threadIdx.x
    t0 = np.int64(cuda.blockIdx.x) * TILE
    base = L + t0

    for idx in range(tid, TILE + 1, TPB):
        j = t0 + idx
        if j < size:
            sh[idx] = acc[j]
        else:
            sh[idx] = 0
    if tid < nsmall:
        q = sq[tid]
        r = base % q
        if r == 0:
            soff[tid] = 0
        else:
            soff[tid] = q - r
    cuda.syncthreads()

    for k in range(nsmall):
        q = sq[k]
        w = sw[k]
        idx = soff[k] + tid * q
        stride = q * TPB
        while idx <= TILE:
            cuda.atomic.add(sh, idx, w)
            idx += stride
    cuda.syncthreads()

    if use_dbg:
        for idx in range(tid, TILE + 1, TPB):
            j = t0 + idx
            if j < size:
                dbg[j] = sh[idx]

    # n >= base in this tile, so log2((base+1)/2) is a lower bound for every n here
    thr = np.int64(math.floor(math.log2((base + 1) * 0.5) * SCALE)) - MARGIN
    for idx in range(tid, TILE, TPB):
        j = t0 + idx
        if j < nn:
            a = sh[idx]
            b = sh[idx + 1]
            if a > 0 and b > 0 and np.int64(a) + np.int64(b) >= thr:
                p = cuda.atomic.add(ocount, 0, 1)
                if p < cap:
                    out[p] = base + idx


# ----------------------------------------------- CPU work on the rare survivors
@njit(cache=True)
def isqrt_nb(y):
    s = np.int64(math.sqrt(y))
    while s * s > y:
        s -= 1
    while (s + 1) * (s + 1) <= y:
        s += 1
    return s


@njit(cache=True)
def rad_fast(x, primes):
    """rad(x) by trial division up to cbrt(x); the cofactor then has at most two
    prime factors, so it is 1, q, q*q' or q^2 (detected as a perfect square)."""
    r = np.int64(1)
    y = x
    for p in primes:
        if p * p > x // p:          # p^3 > x
            break
        if y % p == 0:
            r *= p
            y //= p
            while y % p == 0:
                y //= p
    if y > 1:
        s = isqrt_nb(y)
        if s * s == y:
            r *= s
        else:
            r *= y
    return r


@njit(cache=True)
def rad_td(x, primes):
    """rad(x) by full trial division up to sqrt(x) (independent final check)."""
    r = np.int64(1)
    for p in primes:
        if p * p > x:
            break
        if x % p == 0:
            r *= p
            while x % p == 0:
                x //= p
    if x > 1:
        r *= x
    return r


@njit(cache=True)
def modinv(a, m):
    """Inverse of a mod m (gcd 1, m >= 2); all intermediates stay <= m."""
    a = a % m
    if a == 0:
        return -1
    m0 = m
    x0 = 0
    x1 = 1
    while a > 1:
        q = a // m
        t = m
        m = a % m
        a = t
        t = x0
        x0 = x1 - q * x0
        x1 = t
    if x1 < 0:
        x1 += m0
    return x1


@njit(cache=True)
def process_survivors(surv, primes, res, cands):
    """For each surviving n: exact radicals, CRT partner test, verification.
    res[c] = (kind, m, n) for verified pairs; cands[c] = (n, kind) for every
    CRT candidate (used by the self-tests). Returns (#pairs, #candidates)."""
    npair = 0
    ncand = 0
    for i in range(surv.shape[0]):
        n = surv[i]
        r = rad_fast(n, primes)
        s = rad_fast(n + 1, primes)
        if r == n or s == n + 1:
            continue
        for kind in range(1, 3):
            a = r if kind == 1 else s       # m = 0 (mod a), m = -1 (mod b)
            b = s if kind == 1 else r
            inv = modinv(a, b)
            if inv < 0:
                continue
            k = (b - inv) % b
            if k == 0 or k > (n - 1) // a:
                continue
            if ncand < cands.shape[0]:
                cands[ncand, 0] = n
                cands[ncand, 1] = kind
            ncand += 1
            m = a * k
            step = a * b                    # <= 2n here, cannot overflow
            while m < n:
                if rad_fast(m, primes) == a and rad_fast(m + 1, primes) == b:
                    if npair < res.shape[0]:
                        res[npair, 0] = kind
                        res[npair, 1] = m
                        res[npair, 2] = n
                    npair += 1
                m += step
    return npair, ncand


# ------------------------------------------------------------------- driver
class _Sieve:
    """Device buffers + prime-power tables for blocks of B integers up to `top`."""

    def __init__(self, top, B, cap):
        if B % TILE:
            raise ValueError(f"B must be a multiple of TILE={TILE}")
        self.B = B
        self.cap = cap
        qs, ws = prime_power_table(top)
        small = qs <= TILE
        sq, sw = qs[small], ws[small]
        if len(sq) > TPB:
            raise ValueError("more small prime powers than TPB; raise TPB or lower TILE")
        lq, lw = qs[~small], ws[~small]
        counts = (B + 1) // lq + 1
        nch = (counts + K_CHUNK - 1) // K_CHUNK
        offs = np.cumsum(nch) - nch
        self.nitems = int(nch.sum())
        item_q = np.repeat(lq, nch).astype(np.int64)
        item_w = np.repeat(lw, nch).astype(np.int32)
        item_c = (np.arange(self.nitems) - np.repeat(offs, nch)).astype(np.int32)
        self.nsmall = len(sq)
        self.d_sq = cuda.to_device(sq if len(sq) else np.zeros(1, np.int64))
        self.d_sw = cuda.to_device(sw if len(sw) else np.zeros(1, np.int32))
        self.d_iq = cuda.to_device(item_q if self.nitems else np.zeros(1, np.int64))
        self.d_iw = cuda.to_device(item_w if self.nitems else np.zeros(1, np.int32))
        self.d_ic = cuda.to_device(item_c if self.nitems else np.zeros(1, np.int32))
        self.d_acc = cuda.device_array(B + 1, dtype=np.int32)
        self.d_out = cuda.device_array(cap, dtype=np.int64)
        self.d_ocount = cuda.to_device(np.zeros(1, dtype=np.int64))
        self.d_nodbg = cuda.device_array(1, dtype=np.int32)

    def run_block(self, L, nn, dbg=None):
        """Queue the kernels for n in [L, L+nn). Survivors accumulate in d_out."""
        size = nn + 1
        k_zero[(size + 255) // 256, 256](self.d_acc, size)
        if self.nitems:
            k_big[(self.nitems + TPB_BIG - 1) // TPB_BIG, TPB_BIG](
                np.int64(L), np.int64(size), self.d_iq, self.d_ic, self.d_iw,
                self.nitems, self.d_acc)
        ntiles = (nn + TILE - 1) // TILE
        k_tile[ntiles, TPB](np.int64(L), np.int64(nn), np.int64(size), self.d_acc,
                            self.d_sq, self.d_sw, self.nsmall, self.d_out,
                            self.d_ocount, self.cap,
                            self.d_nodbg if dbg is None else dbg,
                            dbg is not None)

    def take_survivors(self):
        cnt = int(self.d_ocount.copy_to_host()[0])
        if cnt > self.cap:
            raise RuntimeError(f"survivor buffer overflow ({cnt} > {self.cap}); raise cap")
        self.d_ocount.copy_to_device(np.zeros(1, dtype=np.int64))
        if cnt == 0:
            return np.zeros(0, dtype=np.int64)
        return self.d_out[:cnt].copy_to_host()


def _atomic_write(path, text):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        f.write(text)
    os.replace(tmp, path)


def search_hercher(lo, hi, B=1 << 24, outdir=".", cap=1 << 20, quiet=False,
                   checkpoint_every=16, return_candidates=False):
    """Search n in [lo, hi]. Returns (first_kind, second_kind) lists of (m, n);
    with return_candidates=True also returns the sorted list of CRT candidates
    (n, kind), for cross-checking against a reference implementation."""
    lo = max(lo, 2)
    top = hi + 1
    primes = primes_upto(math.isqrt(top) + 2)
    sv = _Sieve(top, B, cap)

    os.makedirs(outdir, exist_ok=True)
    tag = f"hercher_{lo}_{hi}_{B}"
    prog_path = os.path.join(outdir, f"benelux_progress_{tag}.txt")
    res_path = os.path.join(outdir, f"benelux_results_{tag}.txt")
    nxt = lo
    if os.path.exists(prog_path) and not return_candidates:
        nxt = int(open(prog_path).read().strip())
        if not quiet:
            print(f"resuming from {nxt:,}")
    found = []
    if os.path.exists(res_path) and not return_candidates:
        for line in open(res_path):
            found.append(tuple(map(int, line.split())))

    res = np.zeros((4096, 3), dtype=np.int64)
    cands = np.zeros((cap, 2), dtype=np.int64)
    all_cands = []
    t0 = time.perf_counter()
    done0 = nxt
    L = nxt
    nblocks = 0
    while L <= hi:
        nn = min(B, hi + 1 - L)
        sv.run_block(L, nn)
        L += nn
        nblocks += 1
        if nblocks % checkpoint_every == 0 or L > hi:
            surv = sv.take_survivors()          # synchronises with the GPU
            new = []
            if len(surv):
                c, nc = process_survivors(surv, primes, res, cands)
                if c > len(res) or nc > len(cands):
                    raise RuntimeError("result buffer overflow")
                for q in range(c):
                    k, m, n = int(res[q, 0]), int(res[q, 1]), int(res[q, 2])
                    # independent confirmation by full trial division
                    a, b = (rad_td(n, primes), rad_td(n + 1, primes))
                    if k == 2:
                        a, b = b, a
                    if not (rad_td(m, primes) == a and rad_td(m + 1, primes) == b):
                        raise RuntimeError(f"verification mismatch for {(k, m, n)}")
                    new.append((k, m, n))
                if return_candidates:
                    all_cands.extend((int(cands[q, 0]), int(cands[q, 1])) for q in range(nc))
            if not return_candidates:
                with open(res_path, "a") as f:
                    for k, m, n in new:
                        f.write(f"{k} {m} {n}\n")
                _atomic_write(prog_path, str(min(L, hi + 1)))
            found.extend(new)
            if not quiet:
                el = time.perf_counter() - t0
                rate = (min(L, hi + 1) - done0) / max(el, 1e-9)
                eta = (hi + 1 - min(L, hi + 1)) / max(rate, 1e-9)
                print(f"  {min(L, hi + 1):,}/{hi:,}  {rate/1e9:.2f}e9 n/s  "
                      f"eta {eta/60:.1f} min  survivors {len(surv)}"
                      + (f"  new: {new}" if new else ""), flush=True)

    first = sorted({(m, n) for k, m, n in found if k == 1})
    second = sorted({(m, n) for k, m, n in found if k == 2})
    if lo <= 3 <= hi and (2, 3) not in second:   # k=0 family member (m+1 == n)
        second.append((2, 3))
        second.sort()
    if return_candidates:
        return first, second, sorted(set(all_cands))
    return first, second


def debug_weights(L, nn, B=None):
    """Run one block on the GPU and return U(x) for x in [L, L+nn] (for testing)."""
    B = B or ((nn + TILE - 1) // TILE) * TILE
    sv = _Sieve(L + nn, B, 1 << 16)
    d_dbg = cuda.device_array(nn + 1, dtype=np.int32)
    sv.run_block(L, nn, dbg=d_dbg)
    return d_dbg.copy_to_host()

## 3. Self-test (run this first)

Three independent checks; every line should print `PASS`:

1. **Pairs vs brute force** for n ≤ 200,000 (many blocks and tiles).
2. **Sieve exactness**: for whole blocks near 2, 10¹², 2⁴⁰ and 10¹³, the GPU sums U(x) must equal a CPU recomputation *exactly*, and must satisfy 2²⁰·log₂E(x) ≤ U(x) < 2²⁰·log₂E(x) + 64, with E(x) taken from exact radicals.
3. **Candidate lists**: on slices [2, 10⁷], ±10⁷ around n = 1,099,509,530,624 (two family pairs) and the top 2×10⁷ below 10¹³, the GPU's CRT candidates (n, kind) must equal those from a plain CPU exact-radical sieve (the `benelux_cuda` method), and all family pairs in the slice must be found. Candidates are about 10× more numerous than pairs, so this is a much more sensitive test than comparing pairs.

The 10¹³ checks build CPU references, so this cell takes up to a minute or so.

In [ ]:
from collections import defaultdict
import tempfile


def naive(N):
    """Brute-force reference: all pairs (m, n) with m < n <= N."""
    rad = [1] * (N + 3)
    for p in range(2, N + 3):
        if rad[p] == 1:
            for j in range(p, N + 3, p):
                rad[j] *= p
    keys = defaultdict(list)
    for m in range(1, N + 1):
        keys[(rad[m], rad[m + 1])].append(m)
    first, second = set(), set()
    for lst in keys.values():
        for i in range(len(lst)):
            for j in range(i + 1, len(lst)):
                first.add((lst[i], lst[j]))
    for m in range(1, N + 1):
        for n in keys.get((rad[m + 1], rad[m]), []):
            if m < n:
                second.add((m, n))
    return sorted(first), sorted(second)


@njit
def ref_rad_segment(L, size, primes):
    """Exact rad(x) for x in [L, L+size) by a plain CPU segmented sieve."""
    rem = np.arange(L, L + size)
    rad = np.ones(size, dtype=np.int64)
    top = L + size - 1
    for p in primes:
        if p * p > top:
            break
        first = ((L + p - 1) // p) * p
        for j in range(first - L, size, p):
            rad[j] *= p
            v = rem[j] // p
            while v % p == 0:
                v //= p
            rem[j] = v
    for j in range(size):
        if rem[j] > 1:
            rad[j] *= rem[j]
    return rad


@njit
def ref_candidates_nb(L, nn, rad, out):
    """CRT candidates (n, kind) from exact radicals, exactly as benelux_cuda does."""
    c = 0
    for i in range(nn):
        n = L + i
        r = rad[i]
        s = rad[i + 1]
        if r == n or s == n + 1:
            continue
        for kind in range(1, 3):
            a = r if kind == 1 else s
            b = s if kind == 1 else r
            inv = modinv(a, b)
            if inv < 0:
                continue
            k = (b - inv) % b
            if k == 0 or k > (n - 1) // a:
                continue
            out[c, 0] = n
            out[c, 1] = kind
            c += 1
    return c


def ref_candidates(lo, hi, block=1 << 22):
    primes = primes_upto(math.isqrt(hi + 2) + 2)
    out = np.zeros((1 << 16, 2), dtype=np.int64)
    res = []
    L = lo
    while L <= hi:
        nn = min(block, hi + 1 - L)
        rad = ref_rad_segment(L, nn + 1, primes)
        c = ref_candidates_nb(L, nn, rad, out)
        res.extend((int(out[q, 0]), int(out[q, 1])) for q in range(c))
        L += nn
    return sorted(set(res))


@njit
def ref_weights(L, nn, qs, ws):
    """U(x) for x in [L, L+nn], summed on the CPU from the same prime-power table."""
    U = np.zeros(nn + 1, dtype=np.int64)
    for i in range(qs.shape[0]):
        q = qs[i]
        first = ((L + q - 1) // q) * q - L
        for j in range(first, nn + 1, q):
            U[j] += ws[i]
    return U


def family_members(lo, hi):
    """Known infinite-family pairs with lo <= n <= hi (canaries)."""
    f1, f2 = [], []
    k = 2
    while 2 ** (2 * k) - 2 ** (k + 1) <= hi:
        n = 2 ** (2 * k) - 2 ** (k + 1)
        if n >= lo:
            f1.append((2 ** k - 2, n))
        k += 1
    k = 0
    while (2 ** k + 1) ** 2 - 1 <= hi:
        m = 2 ** k + 1
        if m * m - 1 >= lo:
            f2.append((m, m * m - 1))
        k += 1
    return f1, f2


def run_tests(sim=False):
    ok_all = True

    def report(name, ok, extra=""):
        nonlocal ok_all
        ok_all &= bool(ok)
        print(f"{'PASS' if ok else 'FAIL'}  {name}  {extra}", flush=True)

    # 1. pairs against brute force (several blocks, many tiles)
    N = 20_000 if sim else 200_000
    with tempfile.TemporaryDirectory() as d:
        g1, g2 = search_hercher(2, N, B=4 * TILE, outdir=d, quiet=True)
    r1, r2 = naive(N)
    report("pairs vs brute force, n <= %d" % N, g1 == r1 and g2 == r2,
           f"(first {len(g1)}/{len(r1)}, second {len(g2)}/{len(r2)})")

    # 2. GPU sieve sums are exact, and U(x) is a tight upper bound for SCALE*log2(x/rad x)
    if sim:
        cases = [(2, 3 * TILE + 17), (10**7 + 3, 3 * TILE + 17), (2**26 - 5, 2 * TILE + 1)]
    else:
        cases = [(2, (1 << 20) + 17), (10**12 + 12345, (1 << 20) + 17),
                 (2**40 - 777, (1 << 20) + 1), (10**13 - (1 << 21), 1 << 21)]
    for L, nn in cases:
        U_gpu = debug_weights(L, nn).astype(np.int64)
        qs, ws = prime_power_table(L + nn)
        U_ref = ref_weights(L, nn, qs, ws)
        rad = ref_rad_segment(L, nn + 1, primes_upto(math.isqrt(L + nn) + 2))
        x = np.arange(L, L + nn + 1)
        lgE = np.log2((x // rad).astype(np.float64)) * SCALE
        bound_ok = np.all(U_ref >= lgE) and np.all(U_ref - lgE < 64)
        report(f"sieve exact at L={L:,}", np.array_equal(U_gpu, U_ref) and bound_ok)

    # 3. candidate lists identical to an exact-radical reference (incl. huge n)
    if sim:
        ranges = [(2, 20_000), (67092480 - 3000, 67092480 + 3000)]
    else:
        ranges = [(2, 10**7), (1099509530624 - 10**7, 1099509530624 + 10**7),
                  (10**13 - 2 * 10**7, 10**13)]
    for lo, hi in ranges:
        with tempfile.TemporaryDirectory() as d:
            g1, g2, gc = search_hercher(lo, hi, B=4 * TILE if sim else 1 << 22, outdir=d,
                                        quiet=True, return_candidates=True)
        rc = ref_candidates(lo, hi)
        f1, f2 = family_members(lo, hi)
        canary = set(f1) <= set(g1) and set(f2) <= set(g2)
        report(f"candidates [{lo:,}, {hi:,}]", gc == rc and canary,
               f"({len(gc)} GPU vs {len(rc)} ref candidates; {len(f1) + len(f2)} family canaries)")

    print("\nALL TESTS PASSED" if ok_all else "\n*** SOME TESTS FAILED ***")
    return ok_all


run_tests()

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 14 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


PASS  pairs vs brute force, n <= 200000  (first 8/8, second 10/10)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


PASS  sieve exact at L=2  
PASS  sieve exact at L=1,000,000,012,345  
PASS  sieve exact at L=1,099,511,626,999  
PASS  sieve exact at L=9,999,997,902,848  


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


PASS  candidates [2, 10,000,000]  (766 GPU vs 766 ref candidates; 22 family canaries)
PASS  candidates [1,099,499,530,624, 1,099,519,530,624]  (5 GPU vs 5 ref candidates; 2 family canaries)
PASS  candidates [9,999,980,000,000, 10,000,000,000,000]  (0 GPU vs 0 ref candidates; 0 family canaries)

ALL TESTS PASSED


True

## 4. Run

The next cell **prompts for N** (largest n to search) and an optional start. Accepted formats: `1000000000`, `1,000,000,000`, `1e9`, `10**12`. Press Enter for the defaults (N = 10⁹, start = 2). Try 10⁹ first to measure speed.

Pairs not in a known family are labelled `EXCEPTIONAL`; any besides (75, 1215) and (35, 4374) would be new. At the end, the cell checks that every infinite-family member in the range was found (a built-in canary).

### Running very large searches

* Checkpoints go to `OUTDIR` every 16 blocks (≈ 2.7×10⁸ integers). To survive Colab disconnects, mount Google Drive (uncomment the three lines in the cell below).
* **To resume:** re-run the implementation cell, then run the cell below with the **same start, N and `B`**. Checkpoint files are keyed on those values (prefix `benelux_*_hercher_…`, so they never clash with `benelux_cuda` runs).
* **Slices:** use the start prompt to split a big search across sessions, e.g. `2 … 10**12`, then `1000000000001 … 5e12`, etc. Type start values as plain digits.
* **Reference:** for comparison, `benelux_cuda.ipynb` measured about 1e9 integers/s on an A100 (10¹² in about 19 min). Record your own `elapsed` here to compare.
* Memory: about 70 MB of GPU RAM at B = 2²⁴ (the accumulator is 4 bytes per integer).

In [ ]:
import re
import time
from decimal import Decimal

# --- optional: persist checkpoints on Google Drive ---------------------------
# from google.colab import drive
# drive.mount("/content/drive")
# OUTDIR = "/content/drive/MyDrive/benelux_runs"
OUTDIR = "benelux_runs"

B = 1 << 24           # integers per GPU block (multiple of TILE); keep the same when resuming


def parse_count(text, default):
    """Accepts 1000000, 1,000,000, 1e9, 2.5e11 or 10**12. Empty -> default."""
    t = text.strip().replace(",", "").replace("_", "").replace(" ", "")
    if not t:
        return default
    if re.fullmatch(r"\d+", t):
        return int(t)
    m = re.fullmatch(r"(\d+)\*\*(\d+)", t)
    if m:
        return int(m.group(1)) ** int(m.group(2))
    m = re.fullmatch(r"(\d+(?:\.\d+)?)[eE](\d+)", t)
    if m:
        return int(Decimal(m.group(1)) * 10 ** int(m.group(2)))
    raise ValueError(f"Could not understand {text!r}; try 1000000000, 1e9 or 10**12")


HI = parse_count(input("Search up to N (e.g. 10**9, 1e12) [default 10**9]: "), 10**9)
LO = parse_count(input("Start from n (Enter for 2): "), 2)
if LO < 2 or HI < LO:
    raise ValueError(f"Need 2 <= start <= N, got start={LO:,}, N={HI:,}")
print(f"Searching n from {LO:,} to {HI:,}\n")


def is_pow2(x):
    return x > 0 and (x & (x - 1)) == 0


def classify_first(m, n):
    s = m + 2
    if is_pow2(s):
        k = s.bit_length() - 1
        if k >= 2 and n == 2 ** (2 * k) - 2 ** (k + 1):
            return f"infinite family (k={k})"
    return "EXCEPTIONAL"


def classify_second(m, n):
    s = m - 1
    if is_pow2(s):
        k = s.bit_length() - 1
        if k >= 0 and n == m * m - 1:
            return f"infinite family (k={k})"
    return "EXCEPTIONAL"


t0 = time.perf_counter()
p1, p2 = search_hercher(LO, HI, B=B, outdir=OUTDIR)
elapsed = time.perf_counter() - t0
print(f"\nfirst kind: {len(p1)} pairs, second kind: {len(p2)} pairs")
for m, n in p1:
    print(f"  1st ({m}, {n})  [{classify_first(m, n)}]")
for m, n in p2:
    print(f"  2nd ({m}, {n})  [{classify_second(m, n)}]")
print("exceptional first :", [p for p in p1 if classify_first(*p) == "EXCEPTIONAL"])
print("exceptional second:", [p for p in p2 if classify_second(*p) == "EXCEPTIONAL"])
f1, f2 = family_members(LO, HI)
missing = sorted(set(f1) - set(p1)) + sorted(set(f2) - set(p2))
print("family canary check:", "OK" if not missing else f"MISSING {missing}",
      f"({len(f1) + len(f2)} expected family pairs)")
print(f"elapsed: {elapsed:.1f}s  ({(HI - LO + 1) / max(elapsed, 1e-9) / 1e9:.2f}e9 n/s overall)")

Streaming output truncated to the last 5000 lines.
  8,671,538,970,626/10,000,000,000,000  39.42e9 n/s  eta 0.6 min  survivors 0
  8,671,807,406,082/10,000,000,000,000  39.42e9 n/s  eta 0.6 min  survivors 0
  8,672,075,841,538/10,000,000,000,000  39.42e9 n/s  eta 0.6 min  survivors 0
  8,672,344,276,994/10,000,000,000,000  39.42e9 n/s  eta 0.6 min  survivors 0
  8,672,612,712,450/10,000,000,000,000  39.42e9 n/s  eta 0.6 min  survivors 0
  8,672,881,147,906/10,000,000,000,000  39.42e9 n/s  eta 0.6 min  survivors 0
  8,673,149,583,362/10,000,000,000,000  39.42e9 n/s  eta 0.6 min  survivors 0
  8,673,418,018,818/10,000,000,000,000  39.42e9 n/s  eta 0.6 min  survivors 0
  8,673,686,454,274/10,000,000,000,000  39.42e9 n/s  eta 0.6 min  survivors 0
  8,673,954,889,730/10,000,000,000,000  39.42e9 n/s  eta 0.6 min  survivors 0
  8,674,223,325,186/10,000,000,000,000  39.42e9 n/s  eta 0.6 min  survivors 0
  8,674,491,760,642/10,000,000,000,000  39.42e9 n/s  eta 0.6 min  survivors 0
  8,674,760,1